# Radargram Hyperbola Segmentation — Inference Frontend

This notebook is a small frontend for running inference with trained models from this project.

It supports direct prediction for:

- U-Net
- UNet++
- SegFormer
- DINOv3 segmenter
- Mask R-CNN
- Mask2Former
- YOLO11-seg

The output format is the same as `predict.py`:

```text
outputs/notebook_predictions/MODEL_NAME/IMAGE_NAME/
  overlay.png
  mask_all_pred.png
  objects/
    object_001_pred.png
    object_002_pred.png
  coordinates/
    object_001_pixels.csv
    object_002_pixels.csv
  prediction.json
```

Important notes:

1. Use the same config preprocessing that was used during training, especially:
   - `input.image_size`
   - `input.resize_mode`
   - `input.allow_upscale`
   - `input.grayscale`

2. For PyTorch models, predicted masks and CSV pixel coordinates are mapped back to the original image coordinate system.

3. SAM 2 is not included in this generic one-click frontend because SAM 2 needs prompts, such as boxes or points. Use the SAM 2 prompted scripts if you want SAM 2 refinement.

4. For YOLO11-seg, the notebook now checks common Ultralytics save paths, including:
   - `outputs/yolo11_seg/weights/best.pt`
   - `runs/segment/train/weights/best.pt`
   - `runs/segment/train*/weights/best.pt`


## 1. Setup

Run this cell first. It locates the project root, adds `src/` to the Python path, and imports the project utilities.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
import json
import shutil
import sys
from typing import Any

import numpy as np
import torch
from PIL import Image
from IPython.display import display, Markdown, Image as IPyImage, clear_output

# ---------------------------------------------------------------------
# Locate the project root.
# This notebook is expected to be in:
#   radargram_hyperbola_benchmark/notebooks/inference_frontend.ipynb
# but the logic below also works if you start Jupyter from the project root.
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src" / "radarseg").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "src" / "radarseg").is_dir():
        PROJECT_ROOT = candidate

if not (PROJECT_ROOT / "src" / "radarseg").is_dir():
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project root, "
        "or keep this notebook inside the project/notebooks folder."
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Using PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Import inference utilities

This cell defines reusable inference functions. You usually do not need to edit it.

In [ ]:
from radarseg.config import load_config
from radarseg.data.dataset import load_raw_image_for_prediction
from radarseg.external.yolo11 import predict_yolo11_on_image, resolve_yolo_checkpoint
from radarseg.models.factory import build_model
from radarseg.models.mask_rcnn import mask_rcnn_predictions_to_instances
from radarseg.models.mask2former import mask2former_outputs_to_instances
from radarseg.postprocessing.semantic_to_instances import split_semantic_prediction
from radarseg.utils.checkpoint import load_checkpoint
from radarseg.utils.masks import masks_to_boxes
from radarseg.utils.prediction_io import save_prediction_outputs
from radarseg.utils.seed import get_device


def resolve_path(path_like: str | Path) -> Path:
    """Resolve a path relative to PROJECT_ROOT unless it is already absolute."""
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def make_checkpoint_loading_config(cfg: dict[str, Any]) -> dict[str, Any]:
    """Disable external pretrained weights when loading an already-trained checkpoint.

    This avoids unnecessary downloads during notebook inference. Your trained
    weights are loaded from the checkpoint path that you provide.
    """
    cfg = deepcopy(cfg)
    model_cfg = cfg.get("model", {})
    if model_cfg.get("name") == "mask_rcnn":
        model_cfg["pretrained"] = False
    return cfg


def predict_semantic_instances(
    model: torch.nn.Module,
    image_tensor: torch.Tensor,
    *,
    threshold: float,
    min_area: int,
) -> list[dict[str, Any]]:
    """Convert semantic model output into separate object masks.

    Semantic models predict one merged hyperbola mask. Connected-component
    postprocessing then separates it into individual hyperbola instances.
    """
    logits = model(image_tensor[None])
    if logits.shape[1] == 1:
        probs = torch.sigmoid(logits[:, 0])[0]
    else:
        probs = torch.softmax(logits, dim=1)[:, 1][0]

    pred = (probs.detach().cpu().numpy() >= threshold).astype(np.uint8)
    masks = split_semantic_prediction(pred, min_area=min_area)

    instances: list[dict[str, Any]] = []
    if masks:
        boxes = masks_to_boxes(np.stack(masks, axis=0))
        for mask, box in zip(masks, boxes):
            instances.append({
                "score": 1.0,
                "bbox": box.tolist(),
                "mask": mask,
                "area": int(mask.sum()),
            })
    return instances


@dataclass
class LoadedInferenceModel:
    """Container for a loaded model and its config."""
    config_path: Path
    checkpoint_path: Path
    cfg: dict[str, Any]
    model_name: str
    task: str
    device: torch.device | None = None
    model: torch.nn.Module | None = None


_MODEL_CACHE: dict[tuple[str, str], LoadedInferenceModel] = {}


def load_inference_model(config_path: str | Path, checkpoint_path: str | Path) -> LoadedInferenceModel:
    """Load a trained model once and reuse it for repeated predictions."""
    config_path = resolve_path(config_path)
    checkpoint_path = resolve_path(checkpoint_path)

    if not config_path.is_file():
        raise FileNotFoundError(f"Config file not found: {config_path}")
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint file not found: {checkpoint_path}")

    key = (str(config_path.resolve()), str(checkpoint_path.resolve()))
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]

    cfg = load_config(config_path)
    model_name = cfg["model"]["name"]
    task = cfg["model"]["task"]

    if model_name == "sam2":
        raise NotImplementedError(
            "SAM 2 needs prompt boxes or points and is not supported by this generic frontend. "
            "Use scripts/predict_sam2_prompted.py for SAM 2 prompted prediction."
        )

    if model_name == "yolo11_seg":
        # YOLO11 is loaded internally by the Ultralytics API during prediction.
        # Resolve the checkpoint here so missing/incorrect paths are caught early.
        checkpoint_path = resolve_yolo_checkpoint(checkpoint_path)
        key = (str(config_path.resolve()), str(checkpoint_path.resolve()))
        if key in _MODEL_CACHE:
            return _MODEL_CACHE[key]
        loaded = LoadedInferenceModel(
            config_path=config_path,
            checkpoint_path=checkpoint_path,
            cfg=cfg,
            model_name=model_name,
            task=task,
            device=None,
            model=None,
        )
        _MODEL_CACHE[key] = loaded
        return loaded

    num_threads = cfg["training"].get("num_threads")
    if num_threads is not None:
        torch.set_num_threads(int(num_threads))

    device = get_device()
    checkpoint_cfg = make_checkpoint_loading_config(cfg)
    model = build_model(checkpoint_cfg).to(device)
    load_checkpoint(checkpoint_path, model, map_location=device)
    model.eval()

    loaded = LoadedInferenceModel(
        config_path=config_path,
        checkpoint_path=checkpoint_path,
        cfg=cfg,
        model_name=model_name,
        task=task,
        device=device,
        model=model,
    )
    _MODEL_CACHE[key] = loaded
    return loaded


def predict_one_image(
    config_path: str | Path,
    checkpoint_path: str | Path,
    image_path: str | Path,
    output_root: str | Path = "outputs/notebook_predictions",
    *,
    display_result: bool = True,
) -> Path:
    """Run inference on one image and save all prediction outputs.

    Returns
    -------
    Path
        The output directory for this image.
    """
    image_path = resolve_path(image_path)
    output_root = resolve_path(output_root)

    if not image_path.is_file():
        raise FileNotFoundError(f"Image file not found: {image_path}")

    loaded = load_inference_model(config_path, checkpoint_path)
    cfg = loaded.cfg

    threshold = float(cfg["postprocessing"].get("threshold", cfg["model"].get("score_threshold", 0.5)))
    min_area = int(cfg["postprocessing"].get("min_area", 20))

    model_output_root = output_root / loaded.model_name
    output_dir = model_output_root / image_path.stem

    if loaded.model_name == "yolo11_seg":
        instances = predict_yolo11_on_image(
            loaded.checkpoint_path,
            image_path,
            threshold=threshold,
            min_area=min_area,
            imgsz=cfg.get("yolo", {}).get("imgsz"),
        )
        save_prediction_outputs(image_path, instances, output_dir)
        if display_result:
            show_prediction_output(output_dir)
        return output_dir

    assert loaded.model is not None
    assert loaded.device is not None

    image_size = cfg["input"].get("image_size")
    resize_mode = cfg["input"].get("resize_mode", "resize")
    pad_value = int(cfg["input"].get("pad_value", 0))
    allow_upscale = bool(cfg["input"].get("allow_upscale", True))
    grayscale = bool(cfg["input"].get("grayscale", False))

    image_tensor, transform_meta = load_raw_image_for_prediction(
        image_path,
        image_size=image_size,
        grayscale=grayscale,
        resize_mode=resize_mode,
        pad_value=pad_value,
        allow_upscale=allow_upscale,
        return_meta=True,
    )
    image_tensor = image_tensor.to(loaded.device)

    with torch.no_grad():
        if loaded.task == "semantic":
            instances = predict_semantic_instances(
                loaded.model,
                image_tensor,
                threshold=threshold,
                min_area=min_area,
            )
        elif loaded.model_name == "mask_rcnn":
            prediction = loaded.model([image_tensor])[0]
            instances = mask_rcnn_predictions_to_instances(
                prediction,
                threshold=threshold,
                min_area=min_area,
            )
        elif loaded.model_name == "mask2former":
            outputs = loaded.model(image_tensor[None])
            instances = mask2former_outputs_to_instances(
                outputs,
                image_size=tuple(image_tensor.shape[-2:]),
                threshold=threshold,
                min_area=min_area,
            )
        else:
            raise ValueError(
                f"Unsupported model/task combination: model={loaded.model_name}, task={loaded.task}"
            )

    save_prediction_outputs(
        image_path,
        instances,
        output_dir,
        transform_meta=transform_meta,
    )

    if display_result:
        show_prediction_output(output_dir)

    return output_dir


def show_prediction_output(output_dir: str | Path, max_instance_masks: int = 6) -> None:
    """Display saved prediction outputs in the notebook."""
    output_dir = resolve_path(output_dir)

    prediction_json = output_dir / "prediction.json"
    overlay_path = output_dir / "overlay.png"
    mask_all_path = output_dir / "mask_all_pred.png"
    objects_dir = output_dir / "objects"

    if prediction_json.is_file():
        payload = json.loads(prediction_json.read_text(encoding="utf-8"))
        n_instances = len(payload.get("instances", []))
        display(Markdown(f"### Prediction saved to `{output_dir}`"))
        display(Markdown(
            f"- Detected instances: **{n_instances}**  \n"
            f"- Coordinate system: **{payload.get('coordinate_system', 'unknown')}**  \n"
            f"- Original size: **{payload.get('original_image_size', {})}**"
        ))
    else:
        display(Markdown(f"### Prediction saved to `{output_dir}`"))

    if overlay_path.is_file():
        display(Markdown("#### Overlay"))
        display(IPyImage(filename=str(overlay_path), width=1000))

    if mask_all_path.is_file():
        display(Markdown("#### Merged predicted mask"))
        display(IPyImage(filename=str(mask_all_path), width=1000))

    if objects_dir.is_dir():
        mask_paths = sorted(objects_dir.glob("*.png"))[:max_instance_masks]
        if mask_paths:
            display(Markdown(f"#### First {len(mask_paths)} instance mask(s)"))
            for p in mask_paths:
                display(Markdown(f"`{p.name}`"))
                display(IPyImage(filename=str(p), width=700))


print("Inference utilities loaded.")

## 3. Manual prediction mode

Use this if you prefer simple path-based inference.

Edit the three paths below:

- `CONFIG_PATH`
- `CHECKPOINT_PATH`
- `IMAGE_PATH`

Then run the cell.

In [ ]:
# ---------------------------------------------------------------------
# Manual prediction example.
# Change these paths for your trained model and new radargram image.
# ---------------------------------------------------------------------

CONFIG_PATH = "configs/mask_rcnn.yaml"
CHECKPOINT_PATH = "outputs/mask_rcnn/best.pt"
IMAGE_PATH = "dataset/processed/images/example.png"  # Replace this with your new image path.

OUTPUT_ROOT = "outputs/notebook_predictions"

# Uncomment the line below after setting IMAGE_PATH correctly.
# output_dir = predict_one_image(CONFIG_PATH, CHECKPOINT_PATH, IMAGE_PATH, OUTPUT_ROOT)

## YOLO11-seg checkpoint location

If you trained YOLO11-seg and the weights were saved under:

```text
runs/segment/train/weights/best.pt
```

select `yolo11_seg.yaml` in the frontend. The checkpoint field should automatically use that path when the file exists. You can also type it manually:

```text
runs/segment/train/weights/best.pt
```

## 4. Upload-based frontend

Run the next cell to create a small user interface.

You can either:

1. Upload a new image using the upload button, or  
2. Paste an image path in the `Image path` box.

If an uploaded image is present, the uploaded image is used. Otherwise, the notebook uses the path from the `Image path` box.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

def list_config_files() -> list[Path]:
    return sorted((PROJECT_ROOT / "configs").glob("*.yaml"))


def default_checkpoint_for_config(config_path: str | Path) -> str:
    """Guess a checkpoint path for the selected config.

    For YOLO11-seg, Ultralytics may save runs under ``runs/segment/train`` even
    when the project config uses ``outputs/yolo11_seg``. This function checks
    both locations and returns the first existing checkpoint.
    """
    config_path = resolve_path(config_path)
    try:
        cfg = load_config(config_path)
        model_name = cfg["model"]["name"]
        output_dir = resolve_path(cfg["paths"]["output_dir"])

        if model_name == "yolo11_seg":
            candidates: list[Path] = [
                output_dir / "weights" / "best.pt",
                output_dir / "weights" / "last.pt",
                PROJECT_ROOT / "runs" / "segment" / "train" / "weights" / "best.pt",
                PROJECT_ROOT / "runs" / "segment" / "train" / "weights" / "last.pt",
            ]

            # Also check train2, train3, etc. Prefer newest modified checkpoint
            # if several YOLO run folders exist.
            yolo_found = []
            for pattern in [
                "runs/segment/train*/weights/best.pt",
                "runs/segment/train*/weights/last.pt",
                "outputs/yolo11_seg*/weights/best.pt",
                "outputs/yolo11_seg*/weights/last.pt",
            ]:
                yolo_found.extend(PROJECT_ROOT.glob(pattern))

            existing = [p for p in candidates + yolo_found if p.is_file()]
            if existing:
                # Prefer best.pt over last.pt, and among same type choose newest.
                existing = sorted(
                    existing,
                    key=lambda p: (p.name != "best.pt", -p.stat().st_mtime),
                )
                candidate = existing[0]
            else:
                # Reasonable fallback. The run button will show a clearer error
                # if the file does not exist.
                candidate = PROJECT_ROOT / "runs" / "segment" / "train" / "weights" / "best.pt"
        else:
            candidate = output_dir / "best.pt"

        try:
            return str(candidate.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(candidate)
    except Exception:
        return ""


def _get_uploaded_file(upload_widget) -> tuple[str, bytes] | None:
    """Return (filename, content) from ipywidgets FileUpload.

    Handles both ipywidgets 7 and 8 value formats.
    """
    value = upload_widget.value
    if not value:
        return None

    # ipywidgets 8: tuple of dictionaries
    if isinstance(value, tuple):
        item = value[0]
        name = item.get("name", "uploaded_image.png")
        content = item.get("content")
        return name, bytes(content)

    # ipywidgets 7: dictionary keyed by filename
    if isinstance(value, dict):
        name = next(iter(value.keys()))
        item = value[name]
        content = item.get("content")
        return name, bytes(content)

    raise TypeError(f"Unsupported FileUpload value format: {type(value)}")


def save_uploaded_image(upload_widget, upload_dir: str | Path = "notebooks/uploaded_images") -> Path | None:
    uploaded = _get_uploaded_file(upload_widget)
    if uploaded is None:
        return None

    name, content = uploaded
    upload_dir = resolve_path(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)

    # Keep the original suffix when possible.
    suffix = Path(name).suffix.lower()
    if suffix not in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}:
        suffix = ".png"

    path = upload_dir / f"{Path(name).stem}{suffix}"
    path.write_bytes(content)
    return path


if widgets is None:
    display(Markdown(
        "**ipywidgets is not installed.** Use the manual prediction cell above, "
        "or install widgets with `pip install ipywidgets`."
    ))
else:
    config_files = list_config_files()
    if not config_files:
        raise FileNotFoundError(f"No config files found under: {PROJECT_ROOT / 'configs'}")

    config_options = [(p.stem, str(p.relative_to(PROJECT_ROOT))) for p in config_files]
    default_config = "configs/mask_rcnn.yaml" if (PROJECT_ROOT / "configs/mask_rcnn.yaml").is_file() else config_options[0][1]

    config_dropdown = widgets.Dropdown(
        options=config_options,
        value=default_config,
        description="Config:",
        layout=widgets.Layout(width="600px"),
    )

    checkpoint_text = widgets.Text(
        value=default_checkpoint_for_config(default_config),
        description="Checkpoint:",
        layout=widgets.Layout(width="900px"),
    )

    image_path_text = widgets.Text(
        value="",
        description="Image path:",
        placeholder="Optional: path to image if you do not upload a file",
        layout=widgets.Layout(width="900px"),
    )

    upload_widget = widgets.FileUpload(
        accept=".png,.jpg,.jpeg,.tif,.tiff,.bmp",
        multiple=False,
        description="Upload image",
    )

    output_root_text = widgets.Text(
        value="outputs/notebook_predictions",
        description="Output root:",
        layout=widgets.Layout(width="900px"),
    )

    clear_cache_checkbox = widgets.Checkbox(
        value=False,
        description="Clear model cache before prediction",
        indent=False,
    )

    run_button = widgets.Button(
        description="Run prediction",
        button_style="success",
        icon="play",
    )

    output_area = widgets.Output()

    def on_config_change(change):
        if change["name"] == "value":
            checkpoint_text.value = default_checkpoint_for_config(change["new"])

    config_dropdown.observe(on_config_change, names="value")

    def on_run_clicked(_):
        with output_area:
            clear_output()
            try:
                if clear_cache_checkbox.value:
                    _MODEL_CACHE.clear()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                uploaded_path = save_uploaded_image(upload_widget)
                if uploaded_path is not None:
                    image_path = uploaded_path
                    print(f"Using uploaded image: {image_path}")
                else:
                    if not image_path_text.value.strip():
                        raise ValueError("Upload an image or provide an image path.")
                    image_path = resolve_path(image_path_text.value.strip())
                    print(f"Using image path: {image_path}")

                output_dir = predict_one_image(
                    config_dropdown.value,
                    checkpoint_text.value,
                    image_path,
                    output_root_text.value,
                    display_result=True,
                )
                print(f"Done. Output directory: {output_dir}")

            except Exception as exc:
                display(Markdown("### Prediction failed"))
                raise exc

    run_button.on_click(on_run_clicked)

    display(widgets.VBox([
        widgets.HTML("<h3>Radargram Hyperbola Prediction Frontend</h3>"),
        config_dropdown,
        checkpoint_text,
        widgets.HTML("<b>Input image</b>"),
        upload_widget,
        image_path_text,
        output_root_text,
        clear_cache_checkbox,
        run_button,
        output_area,
    ]))

## 5. Inspect a saved prediction

Use this cell to redisplay a prediction folder that was already created.

In [ ]:
# Example:
# show_prediction_output("outputs/notebook_predictions/mask_rcnn/example")

# Change this path if you want to inspect an existing output folder:
OUTPUT_DIR_TO_INSPECT = ""

if OUTPUT_DIR_TO_INSPECT:
    show_prediction_output(OUTPUT_DIR_TO_INSPECT)
else:
    print("Set OUTPUT_DIR_TO_INSPECT to a prediction folder path, then rerun this cell.")